## 회원 CRUD - auth.users, user_details

- `AuthUsers`: 회원가입, 로그인, 로그아웃, 본인 이메일/비밀번호 수정
- `UserService`: 회원 상세정보 목록 조회, 수정, 소프트 삭제
- 두 클래스는 외부에서 전달받은 같은 `supabase` 객체를 사용한다.
- `deleted_at IS NULL`: 삭제되지 않은 상세정보. 일반 조회에 사용한다.
- `deleted_at IS NOT NULL`: 삭제 처리된 상세정보. 삭제 이력을 조회할 때 사용한다.


In [ ]:
import os
import uuid
from datetime import datetime, timezone
from dotenv import load_dotenv
from supabase import create_client

# 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Supabase 클라이언트 생성
supabase = create_client(url, key)

## AuthUsers: 회원 인증과 본인 정보

전체 회원 상세정보 조회는 아래 `UserService.get_active_users()`를 사용하므로 `get_users()`를 중복 구현하지 않는다. Auth 계정 자체의 관리자 조회는 별도 기능이다.

`get_my_info()`는 삭제되지 않은 본인 상세정보와 이메일 원문을 반환한다. 상세정보가 없거나 삭제되었으면 None이다. 이메일 마스킹은 발표 화면이나 다른 사용자에게 보여주는 화면에서 필요에 따라 적용한다.

`resign()`은 본인의 `user_details.deleted_at`에 시간을 기록한다. Auth 계정 삭제, 세션 종료, 재로그인 차단까지 수행하는 메서드는 아니다. `is_logged_in()`은 저장된 사용자 객체의 존재를 검사하며 관리자 권한이나 계정 활성 상태를 별도로 검증하지 않는다.


In [ ]:
class AuthUsers:

    def __init__(self,supabase):
        self.supabase = supabase
        self.user = None

        try:
            response = (
                self.supabase.auth.get_user()
            )
            
            self.user = response.user

        except Exception:
            print("회원 정보가 없습니다.")

    
    # [CREATE] 회원가입
    def sign_up(self, mail, pw):
        response = (
            self.supabase.auth.sign_up({
                "email":mail,
                "password":pw
            })
        )

        return response

    # [AUTH] 로그인
    def sign_in(self, mail, pw):
        response = (
            self.supabase.auth.sign_in_with_password({
                "email": mail,
                "password": pw
            })
        )

        self.user = response.user

        return response

    # [UPDATE] 회원 기본정보 수정 
    # auth.users에서 수정 가능한 이메일 또는 비밀번호 변경
    # 회원 상세 정보 수정과 별도 기능으로 구현
    def update(self, data):
        self.check_authority()

        allowed_fields = {"email","password"}
        invalid_fields = set(data.keys()) - allowed_fields

        if invalid_fields:
            raise ValueError ("해당 정보는 [회원 상세 정보]에서 수정할 수 있습니다.")

        response = (
            self.supabase.auth.update_user(data)
            )

        self.user = response.user

        return response

    # [DELETE] 회원 탈퇴
    # 실제 데이터를 삭제하지 않고 deleted_at을 기록하는 Soft Delete
    def resign(self):
        self.check_authority()
        response = (
            self.supabase.table("user_details")
            .update({
                "deleted_at": datetime.now(timezone.utc)
                .isoformat()
            })
            .eq("id", self.user.id)
            .is_("deleted_at", "null")
            .execute()
            )
        
        return response.data
        
        
    # [AUTH] 로그아웃
    def sign_out(self):
        self.check_authority()
        response = (
            self.supabase.auth.sign_out()
        )

        self.user = None

        return response

     # [READ] 현재 로그인한 회원 정보 조회
    def get_my_info(self):
        self.check_authority()
        response = (
            self.supabase.table("user_details")
                .select("*")
                .eq("id", self.user.id)
                .is_("deleted_at", "null")
                .maybe_single()
                .execute()
        )
        if response is not None and response.data:
            # auth.users의 이메일을 일부 마스킹하여 추가
            # email = self.user.email
            # name, domain = email.split("@")

            # masked_email = (
            #     name[:2]
            #     + "*" * max(len(name) - 2, 1)
            #     + "@"
            #     + domain
            # )

            # data["email"] = masked_email
            data = dict(response.data)
            data['email'] = self.user.email

            return data

        return None 

    # user_details에 전체 조회기능이 있음 - 이건 아래 출력 포맷(?) 참고용으로 남겨둠
    # def get_users(self, page = 1, limit = 20, *, search = None):
    #     pass


    # 현재 로그인 여부 확인
    def is_logged_in(self):
        if self.user: 
            return True

        return False

    # 로그인 권한 확인
    def check_authority(self):
        if not self.is_logged_in():
            raise PermissionError("로그인이 필요합니다.")


**[UserService 클래스 참고사항]**
- 아래 str 자료형은 Python 기준으로 작성되었습니다. (실제 테이블은 text 자료형 생성)
- `id` 타입힌팅: 이 변수 자리에는 str이나 uuid.UUID 객체 둘 다 사용할 수 있습니다.
- AuthUsers 클래스 내 회원정보를 전달하는 앱함수가 없다는 전제 하에  
  AuthUsers 클래스에서 생성된 회원정보를 가지고 오기 위한 SQL 트리거가 작동합니다.  
  (AX Team Project Supabase Shared SQL Query - user_details table 참조)
- 트리거로 AuthUsers 클래스에서 생성된 회원정보를 user_details 테이블에 가져오기 때문에  
  아래 create_user_detail 함수는 사용하지 않습니다. (주석 처리)  
  ※ 해당 함수가 살아있을 경우 SQL 트리거 설계와 충돌 여지가 있으므로 유의해야 합니다.  
- 별도 에러 핸들링은 없습니다... 그래서 빈 리스트가 반환되거나 에러가 발생할 수 있습니다.  
- AuthUsers 클래스 (서인님 코드) 참고해서 의존성 주입 패턴으로 수정, 반영했습니다.  
  ※ 의존성 주입 패턴: 객체를 외부에서 한 번만 만들고, 클래스 내부로 전달 받는 패턴

In [ ]:
class UserService:

    def __init__(self, supabase):
        self.supabase = supabase    # Supabase 클라이언트 객체 받기
        self.table = "user_details" # 사용할 테이블

    # 1. 회원 상세 정보 등록 (Create)
    # def create_user_detail(
    #         self,
    #         id: str|uuid.UUID,   # PK + refer.auth.users(id)
    #         type: str = "BUYER", # SELLER or BUYER
    #         phone: str = None,   # auth의 phone이 아닌 단순 메타데이터
    #         zipcode: str = None,
    #         address: str = None,
    #         address_sub: str = None
    # ):

    #     data = {
    #         "id": id,
    #         "type": type,
    #         "phone": phone,
    #         "zipcode": zipcode,
    #         "address": address,
    #         "address_sub": address_sub,      # 트레일링 콤마
    #     }                                    # 데이터를 받습니다.

    #     response = (
    #         self.supabase.table(self.table)
    #                      .insert(data)
    #                      .execute()          # 받은 데이터를 DB에 저장합니다.
    #     )

    #     return response.data

    # 2. 삭제되지 않은 회원 전체 조회 (Read: Deleted_at IS NULL)
    # supabase-py의 .is_ 매서드는 파이썬의 `IS` 연산자와의 충돌을 막기 위해 언더바를 사용합니다.
    def get_active_users(self):
        response = (
            self.supabase.table(self.table)
                         .select("*")
                         .is_("deleted_at", "null") # deleted_at 열의 null 값을 탐색합니다.
                         .execute()
        )

        return response.data

    # 3. 삭제되지 않은 단일 회원 조회
    def get_active_user_by_id(self, id: str|uuid.UUID):
        response = (
            self.supabase.table(self.table)
                         .select("*")
                         .eq("id", id)
                         .is_("deleted_at", "null")
                         .execute()
        )

        return response.data

    # 4-1. 수정 가능한 회원 정보 필드
    updatable_fields = {"type", "phone", "zipcode", "address", "address_sub"}

    # 4-2. 회원 정보 수정 (Update)
    def update_user_detail(self, id: str|uuid.UUID, **kwargs):
        invalid_keys = set(kwargs.keys()) - self.updatable_fields

        if invalid_keys:
            raise ValueError(f"수정할 수 없는 필드입니다.:{invalid_keys}")

        kwargs["modified_at"] = datetime.now(timezone.utc).isoformat()
        response = (
            self.supabase.table(self.table)
                         .update(kwargs)             # 수정할 회원정보를 가변키워드인자로 받습니다.
                         .eq("id", id)               # 수정할 회원정보를 가진 회원이 맞는지 확인합니다.
                         .is_("deleted_at", "null")  # 삭제되지 않은 회원임을 확인합니다. 
                         .execute()
        )

        return response.data

    # 5. 소프트 삭제 (deleted_at 업데이트)
    def soft_delete_user(self, id: str|uuid.UUID):
        now = datetime.now(timezone.utc).isoformat() # 현재 시간을 받습니다. 
        response = (
            self.supabase.table(self.table)
                         .update({
                             "deleted_at":now,            # 받았던 현재시간을 deleted_at 열에 업데이트합니다. ← 삭제 역할!
                             "modified_at":now           # 삭제도 수정이므로 modified_at 열을 업데이트합니다.(정책에 따름)
                         })
                         .eq("id", id)                    # 삭제할 회원정보를 가진 회원이 맞는지 확인합니다. 
                         .is_("deleted_at", "null")       # 삭제되지 않은 회원임을 확인합니다. (이제 삭제할겁니다.)
                         .execute()
        )
        return response.data